# Guide intelligent du Musée · 03 Diagnostic (jalon `m2`)

Suivre l'entraînement du réseau dense de `02_baseline`, diagnostiquer ses défauts à partir des courbes, les corriger un par un, puis analyser ses erreurs.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import math

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader

from commun import (MODELS_DIR, SEED, UNKNOWN_ARTIST, ArtworkDataset, DenseNet, fit, get_device, load_dense,
                    load_images, load_metadata, normalization_stats, predict, save_dense, set_seed,
                    split_indices, standardize)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Courbes de suivi et diagnostic

Le réseau de référence de `02_baseline` (section 3.4) est réentraîné à l'identique pour suivre, à chaque époque, la perte et l'accuracy en train et en validation : pixels aplatis, une couche cachée de 128 neurones en sigmoïde, SGD avec un taux d'apprentissage de 0,1, batch de 64, 30 époques, graine 0, sur CPU. Le test n'est pas utilisé.

In [ ]:
N_IN = 3 * 128 * 128
HIDDEN = [128]
LR = 0.1
BATCH_SIZE = 64
N_EPOCHS = 30
CPU = torch.device("cpu")
LN_C = math.log(len(CLASSES))  # perte d'un réseau qui répond au hasard : ln 5

datasets = {k: ArtworkDataset(X_all, y_all, splits[k], MEAN, STD) for k in ("train", "val")}
val_loader = DataLoader(datasets["val"], batch_size=256)
loss_fn = nn.CrossEntropyLoss()


def train_dense(hidden: list[int], lr: float, seed: int = SEED) -> tuple[DenseNet, dict[str, list[float]]]:
    """Entraîne un réseau dense par SGD et renvoie le modèle et son historique par époque."""
    set_seed(seed)
    model = DenseNet(N_IN, hidden, len(CLASSES))
    train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    return model, fit(model, train_loader, val_loader, loss_fn, optimizer, N_EPOCHS, CPU)


def summarize(history: dict[str, list[float]]) -> dict[str, float]:
    """Chiffres du diagnostic : valeurs à la dernière époque et minimum de la perte de validation."""
    best = int(np.argmin(history["val_loss"]))
    return {"perte_train": history["train_loss"][-1], "perte_val": history["val_loss"][-1],
            "acc_train": history["train_acc"][-1], "acc_val": history["val_acc"][-1],
            "epoque_min_perte_val": best + 1, "min_perte_val": history["val_loss"][best]}

### 2.1 Réseau de référence

In [ ]:
baseline, history = train_dense(HIDDEN, LR)
ref = summarize(history)

print(f"Dernière époque : perte {ref['perte_train']:.3f} en train, {ref['perte_val']:.3f} en validation "
      f"(écart {ref['perte_val'] - ref['perte_train']:.3f}) ; accuracy {ref['acc_train']:.3f} en train, "
      f"{ref['acc_val']:.3f} en validation (écart {ref['acc_train'] - ref['acc_val']:.3f})")
print(f"Minimum de la perte de validation : {ref['min_perte_val']:.3f} à l'époque {ref['epoque_min_perte_val']} ; "
      f"perte d'un réseau qui répond au hasard : ln 5 = {LN_C:.3f}")
print(f"Meilleure accuracy de validation : {max(history['val_acc']):.3f} à l'époque {int(np.argmax(history['val_acc'])) + 1}")

In [ ]:
# Palette de référence (créneaux 1 et 2), encre et grille discrètes
COLORS = {"train": "#2a78d6", "val": "#eb6834"}
LABELS = {"train": "train", "val": "validation"}
INK, MUTED = "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 11, "font.size": 10,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.5,
})


def plot_curves(ax, history: dict[str, list[float]], metric: str, chance: float) -> None:
    """Courbes train et validation d'une métrique ('loss' ou 'acc'), étiquetées en bout de ligne,
    avec en pointillés la valeur d'un réseau qui répond au hasard."""
    epochs = np.arange(1, len(history[f"train_{metric}"]) + 1)
    ends = {split: history[f"{split}_{metric}"][-1] for split in ("train", "val")}
    low, high = sorted(ends, key=ends.get)
    for split in ("train", "val"):
        ax.plot(epochs, history[f"{split}_{metric}"], color=COLORS[split], label=LABELS[split])
        nudge = -6 if split == low else 6  # écarte les deux étiquettes quand les courbes finissent proches
        ax.annotate(LABELS[split], (epochs[-1], ends[split]), xytext=(5, nudge), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    ax.axhline(chance, color=MUTED, linestyle="--", linewidth=1, label="réponse au hasard")
    ax.set_xlabel("époque")
    ax.set_xlim(1, epochs[-1] * 1.18)  # place pour les étiquettes en bout de ligne
    ax.set_xticks([1, *range(5, epochs[-1] + 1, 5)])


fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
plot_curves(ax_loss, history, "loss", LN_C)
e = ref["epoque_min_perte_val"]
ax_loss.plot(e, ref["min_perte_val"], "o", color=COLORS["val"], markersize=6)
ax_loss.annotate(f"minimum, époque {e}", (e, ref["min_perte_val"]), xytext=(8, -20), textcoords="offset points",
                 color=INK, fontsize=9)
ax_loss.set_ylabel("perte (entropie croisée)")
ax_loss.set_title("Perte")
plot_curves(ax_acc, history, "acc", 1 / len(CLASSES))
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
fig.legend(*ax_loss.get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("Réseau de référence, par époque", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

**Diagnostic du réseau de référence : sur-apprentissage, dès les premières époques.**

- La perte de train descend régulièrement jusqu'à 0,499, mais la perte de validation atteint son minimum dès l'époque 4 (1,641), puis remonte en oscillant pour finir à 1,955. À la dernière époque, l'écart entre train et validation atteint 1,457 en perte et 0,503 en accuracy (0,850 contre 0,347) : le réseau apprend le train par cœur au lieu de généraliser.
- La perte de validation finit au-dessus de ln 5 (1,955 contre 1,609) : le réseau fait pire, en perte, qu'une réponse uniforme. Ses erreurs sont commises avec assurance, et l'entropie croisée les punit fortement, alors que son accuracy reste au-dessus du hasard (0,347 contre 0,20).
- Perte et accuracy de validation ne désignent pas la même époque : minimum de perte à l'époque 4, meilleure accuracy (0,360) à l'époque 18. Avec 300 œuvres, les deux courbes de validation sont très bruitées d'une époque à l'autre.
- Le taux d'apprentissage n'est pas trop fort : la perte de train descend sans sauts, contrairement au premier échec provoqué ci-dessous.

### 2.2 Échecs provoqués

Trois runs qui ne changent qu'une chose par rapport à la référence, pour reconnaître sur les courbes les pannes classiques d'un entraînement :

- **taux d'apprentissage trop fort** : 3 au lieu de 0,1 ;
- **taux d'apprentissage trop faible** : 0,001 au lieu de 0,1 ;
- **réseau profond en sigmoïde** : 6 couches cachées de 128 neurones au lieu d'une.

In [ ]:
FAILURES = {
    "lr trop fort (3)": (HIDDEN, 3.0),
    "lr trop faible (0,001)": (HIDDEN, 0.001),
    "6 couches en sigmoïde": ([128] * 6, LR),
}
failure_histories = {name: train_dense(hidden, lr)[1] for name, (hidden, lr) in FAILURES.items()}

table = pd.DataFrame({"référence (lr 0,1)": ref} | {name: summarize(h) for name, h in failure_histories.items()}).T
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(failure_histories), figsize=(13, 3.6))
for ax, (name, h) in zip(axes, failure_histories.items()):
    plot_curves(ax, h, "loss", LN_C)
    ax.set_title(name)
axes[0].set_ylabel("perte (entropie croisée)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("Échecs provoqués : perte par époque (échelles différentes)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

Pour le réseau profond, la norme du gradient de chaque couche au premier batch, avant tout entraînement, montre où le signal se perd :

In [ ]:
def gradient_norms(hidden: list[int]) -> list[float]:
    """Norme du gradient des poids de chaque couche Linear, de l'entrée vers la sortie, sur un batch."""
    set_seed()
    model = DenseNet(N_IN, hidden, len(CLASSES))
    xb, yb = next(iter(DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)))
    loss_fn(model(xb), yb).backward()
    return [layer.weight.grad.norm().item() for layer in model.net if isinstance(layer, nn.Linear)]


deep_norms, ref_norms = gradient_norms([128] * 6), gradient_norms(HIDDEN)
grads = pd.DataFrame({"6 couches en sigmoïde": deep_norms,
                      "référence": ref_norms + [None] * (len(deep_norms) - len(ref_norms))},
                     index=[f"couche {i}" for i in range(1, len(deep_norms) + 1)])
print(f"Réseau profond : gradient de la 1re couche {deep_norms[0]:.1e}, de la sortie {deep_norms[-1]:.1e}, "
      f"soit {deep_norms[-1] / deep_norms[0]:.0f} fois plus petit à l'entrée")
grads.map(lambda v: f"{v:.1e}" if pd.notna(v) else "")

**Diagnostic des échecs provoqués**

- **Taux d'apprentissage trop fort (3)** : les pertes restent très au-dessus de ln 5 (19,6 en train et 33,5 en validation à la dernière époque) et sautent d'une époque à l'autre : chaque pas de gradient dépasse le minimum au lieu de s'en approcher. L'accuracy reste au niveau du hasard (0,273 en validation). Pas de `nan` sur ces 30 époques, mais le remède est le même : réduire le taux d'apprentissage.
- **Taux d'apprentissage trop faible (0,001)** : les deux pertes descendent lentement, sans plateau ; la perte de train vaut encore 1,346 après 30 époques, l'entraînement n'est pas fini. Mais l'écart entre train et validation reste faible (0,508 contre 0,410 en accuracy), et l'accuracy de validation dépasse celle de la référence (0,410 contre 0,347) : apprendre lentement freine le sur-apprentissage, comme un arrêt précoce. Ce résultat porte sur une seule graine et doit être confirmé sur trois ; la grille de `02_baseline` (0,01 à 0,3) n'allait pas aussi bas.
- **6 couches en sigmoïde** : la perte de train reste plate autour de ln 5 (1,627 à la dernière époque) et l'accuracy au niveau du hasard (0,200 en validation) : le réseau n'apprend rien. Les normes de gradient montrent pourquoi : avant tout entraînement, le gradient de la première couche est environ 5 800 fois plus petit que celui de la couche de sortie. La dérivée de la sigmoïde vaut au plus 1/4, et la rétropropagation la multiplie à chaque couche : le gradient s'évanouit en remontant vers l'entrée. Remèdes vus en séance 5 : ReLU et l'initialisation de He.

### 2.3 Bilan

- **Le réseau de référence sur-apprend** : la perte de validation est minimale dès l'époque 4, et l'écart entre train et validation dépasse 0,5 en accuracy à la dernière époque. Son problème est la généralisation, pas l'optimisation.
- **Chaque panne a sa signature sur les courbes** : une perte qui saute très au-dessus de ln 5 (taux trop fort), une perte qui descend lentement sans plateau (taux trop faible), une perte plate à ln 5 (réseau qui n'apprend rien, ici par évanouissement du gradient).
- **Pistes pour les corrections de la section 3** : régularisation (weight decay, dropout), arrêt précoce, et un taux d'apprentissage plus faible, qui donne 0,410 en validation sur une graine.

## 3. Corrections testées une par une

Le diagnostic de la section 2 désigne la généralisation comme le problème du réseau de référence. Chaque correction est testée seule, un changement à la fois, sur 3 graines. L'accuracy de validation est relevée à la dernière époque (à la meilleure époque avec l'arrêt précoce), et une correction n'est retenue que si son gain dépasse nettement l'écart-type entre graines.

Deux tours : le premier part de la référence, le second de la meilleure correction du premier. Le réseau retenu est choisi sur la validation ; le test n'est pas utilisé.

In [ ]:
SEEDS = [0, 1, 2]


def train_config(seed: int, lr: float = LR, optimizer: str = "sgd", momentum: float = 0.0,
                 weight_decay: float = 0.0, n_epochs: int = N_EPOCHS, patience: int | None = None,
                 **net) -> tuple[DenseNet, dict]:
    """Entraîne un réseau dense qui ne diffère de la référence que par la configuration passée."""
    set_seed(seed)
    model = DenseNet(N_IN, HIDDEN, len(CLASSES), **net)
    if optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
    return model, fit(model, train_loader, val_loader, loss_fn, opt, n_epochs, CPU, patience=patience)


def compare(configs: dict[str, dict]) -> pd.DataFrame:
    """Une ligne par configuration, en moyenne sur les graines, à l'époque retenue :
    la dernière, ou la meilleure avec l'arrêt précoce."""
    rows = []
    for name, config in configs.items():
        runs = []
        for seed in SEEDS:
            _, h = train_config(seed, **config)
            e = h.get("best_epoch", len(h["val_acc"])) - 1
            runs.append({"acc_train": h["train_acc"][e], "acc_val": h["val_acc"][e], "epoques": e + 1,
                         "nan": any(math.isnan(x) for x in h["train_loss"])})
        runs = pd.DataFrame(runs)
        rows.append({"correction": name, "acc_train": runs["acc_train"].mean(), "acc_val": runs["acc_val"].mean(),
                     "acc_val_std": runs["acc_val"].std(), "ecart": runs["acc_train"].mean() - runs["acc_val"].mean(),
                     "epoques": runs["epoques"].mean(), "nan": runs["nan"].any()})
    return pd.DataFrame(rows).set_index("correction")

### 3.1 Premier tour : une correction à la fois sur la référence

La référence : sigmoïde, SGD avec un taux d'apprentissage de 0,1, 30 époques, sans régularisation. La colonne `nan` signale un entraînement qui a divergé.

In [ ]:
ROUND_1 = {
    "référence (SGD, lr 0,1)": {},
    "lr 0,001": dict(lr=0.001),
    "ReLU + init. de He": dict(activation=nn.ReLU, he_init=True),
    "weight decay 1e-2": dict(weight_decay=1e-2),
    "dropout 0,5": dict(dropout=0.5),
    "arrêt précoce (patience 5)": dict(patience=5),
    "batch norm": dict(batch_norm=True),
    "momentum 0,9": dict(momentum=0.9),
    "Adam (lr 1e-3)": dict(optimizer="adam", lr=1e-3),
}
round_1 = compare(ROUND_1)
round_1.round(3)

- **Un seul gain net : le taux d'apprentissage.** À 0,001 au lieu de 0,1, l'accuracy de validation passe de 0,302 ± 0,053 à 0,419 ± 0,008, et l'écart entre train et validation de 0,531 à 0,086. Les résultats deviennent aussi stables d'une graine à l'autre : l'écart-type est divisé par plus de six.
- **Adam aide aussi** (0,386 ± 0,018), mais moins ; il change à la fois l'optimiseur et le pas.
- **Les régularisations ne compensent pas un pas trop grand** : le weight decay (0,310), le dropout (0,307) et le momentum (0,293) restent dans le bruit de la référence ; l'arrêt précoce s'arrête en moyenne dès l'époque 4, à 0,301 ; la batch norm fait pire (0,216).
- **ReLU et l'initialisation de He divergent** : avec un pas de 0,1, la perte part en `nan` et l'accuracy tombe au niveau du hasard (0,200). La sigmoïde, bornée, masquait ce symptôme du taux trop fort en section 2.

### 3.2 Second tour : une correction à la fois sur lr 0,001

In [ ]:
BASE_2 = dict(lr=0.001)
ROUND_2 = {
    "lr 0,001, base du second tour": {},
    "+ 60 époques": dict(n_epochs=60),
    "+ ReLU + init. de He": dict(activation=nn.ReLU, he_init=True),
    "+ weight decay 1e-2": dict(weight_decay=1e-2),
    "+ dropout 0,5": dict(dropout=0.5),
    "+ arrêt précoce (patience 5, 100 époques max.)": dict(n_epochs=100, patience=5),
    "+ batch norm": dict(batch_norm=True),
    "+ momentum 0,9": dict(momentum=0.9),
}
round_2 = compare({name: BASE_2 | config for name, config in ROUND_2.items()})
round_2.round(3)

- **Aucune correction ne bat lr 0,001 seul** : 60 époques au lieu de 30 et le weight decay laissent l'accuracy de validation à 0,419 ; le dropout et l'arrêt précoce (0,410 chacun) restent dans l'écart-type.
- **Tout ce qui accélère l'apprentissage fait revenir le sur-apprentissage** : avec ReLU et l'initialisation de He, 0,973 en train mais 0,366 en validation ; avec le momentum, 0,836 en train et 0,370 en validation. La batch norm reste la pire (0,331).
- À ce pas, le réseau apprend lentement et finit ses 30 époques avant d'apprendre le train par cœur.

### 3.3 Réseau retenu

Le meilleur score moyen de validation sur les deux tours ; à égalité, la configuration la plus simple, c'est-à-dire la première du tableau. Il est réentraîné avec la graine 0 et comparé à la référence de la section 2, puis à la régression logistique sauvegardée par `02_baseline`.

In [ ]:
CONFIGS = ROUND_1 | {name: BASE_2 | config for name, config in ROUND_2.items()}
results = pd.concat([round_1, round_2])
best_name = results["acc_val"].round(4).idxmax()  # première ligne en cas d'égalité
best_model, best_history = train_config(SEED, **CONFIGS[best_name])
best = summarize(best_history)

logreg = joblib.load(MODELS_DIR / "regression_logistique.joblib")
X_val_flat = standardize(X_all[splits["val"]], MEAN, STD).flatten(1).numpy()
acc_logreg = logreg.score(X_val_flat, y_all[splits["val"]].numpy())

reference = results.loc["référence (SGD, lr 0,1)"]
print(f"Réseau retenu : {best_name}")
print(f"Accuracy de validation sur 3 graines : {results.loc[best_name, 'acc_val']:.3f} ± {results.loc[best_name, 'acc_val_std']:.3f}, "
      f"contre {reference['acc_val']:.3f} ± {reference['acc_val_std']:.3f} pour la référence")
print(f"Graine {SEED}, dernière époque : accuracy {best['acc_train']:.3f} en train, {best['acc_val']:.3f} en validation "
      f"(écart {best['acc_train'] - best['acc_val']:.3f}) ; minimum de la perte de validation à l'époque {best['epoque_min_perte_val']}")
print(f"Régression logistique de 02_baseline, rechargée : {acc_logreg:.3f} en validation")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.8), sharey="col")
for row, (title, h) in enumerate([("référence (lr 0,1)", history), (f"retenu ({best_name})", best_history)]):
    plot_curves(axes[row, 0], h, "loss", LN_C)
    plot_curves(axes[row, 1], h, "acc", 1 / len(CLASSES))
    axes[row, 0].set_ylabel("perte (entropie croisée)")
    axes[row, 1].set_ylabel("accuracy")
    axes[row, 0].set_title(f"{title} : perte")
    axes[row, 1].set_title(f"{title} : accuracy")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("Référence et réseau retenu, par époque (mêmes échelles)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

### 3.4 Sauvegarde

Le réseau retenu est sauvegardé avec sa configuration dans `modeles/`, qui n'est pas versionné : les notebooks du ResNet et de la recommandation le reconstruisent sans la connaître.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
dense_path = MODELS_DIR / "reseau_dense.pt"
save_dense(best_model, dense_path)

reloaded = load_dense(dense_path)
assert torch.equal(predict(reloaded, val_loader, CPU), predict(best_model, val_loader, CPU))
print(f"{dense_path.name} : {dense_path.stat().st_size / 1e6:.1f} Mo, configuration {reloaded.config}")
print("Prédictions identiques après rechargement")

### 3.5 Bilan

- **Réseau dense retenu : sigmoïde, SGD avec un taux d'apprentissage de 0,001, 30 époques, sans régularisation.** Il atteint 0,419 ± 0,008 en validation sur 3 graines, contre 0,302 ± 0,053 pour la référence. La correction décisive est le pas d'apprentissage, divisé par 100.
- **Le sur-apprentissage a disparu des courbes** : avec la graine 0, 0,508 en train contre 0,410 en validation (un écart de 0,098 au lieu de 0,503), et la perte de validation descend jusqu'à l'époque 26 au lieu de remonter dès l'époque 4.
- **Le réseau dense ne fait pas mieux que la régression logistique** (0,427 en validation) : l'écart, inférieur à un point, est bien en deçà de l'incertitude de mesure (environ ± 5,6 points). Sur des pixels bruts, la couche cachée n'apporte rien qu'un modèle linéaire régularisé ne sache faire. Hypothèse pour le jalon `m3` : reconnaître un style demande des motifs détectés où qu'ils soient dans l'image, ce que permettent les convolutions du ResNet.
- **Le réseau retenu est sauvegardé** (25,2 Mo) pour la comparaison avec le ResNet et l'évaluation finale.

## 4. Analyse des erreurs

Le réseau retenu à la section 3 (graine 0) est analysé sur les œuvres de validation : quels styles il confond, si les artistes inconnus et les genres dominants y jouent un rôle, et sur quoi il s'appuie. Le test n'est pas utilisé.

### 4.1 Styles confondus

Une ligne par style réel, une colonne par style prédit. Une paire de styles additionne les erreurs dans les deux sens.

In [ ]:
y_val = y_all[splits["val"]].numpy()
y_pred = predict(best_model, val_loader, CPU).numpy()
val_meta = meta.iloc[splits["val"]].reset_index(drop=True)
val_meta = val_meta.assign(pred=[CLASSES[p] for p in y_pred], correct=y_pred == y_val)

cm = pd.DataFrame(confusion_matrix(y_val, y_pred), index=CLASSES, columns=CLASSES)
cm.index.name, cm.columns.name = "style réel", "style prédit"
errors = cm.where(~np.eye(len(CLASSES), dtype=bool)).stack().dropna().astype(int)
pairs = errors.groupby(lambda pair: " et ".join(sorted(pair))).sum().sort_values(ascending=False)

print(f"{errors.sum()} œuvres mal classées sur {len(y_val)}")
print("Accuracy par style :", ", ".join(f"{s} {cm.loc[s, s] / cm.loc[s].sum():.2f}" for s in CLASSES))
print("Erreurs par style prédit :", ", ".join(f"{s} {n}" for s, n in errors.groupby(level=1).sum().items()))
print("Erreurs les plus fréquentes :", ", ".join(f"{a} pris pour {b} {n}" for (a, b), n in errors.nlargest(3).items()))
print("Paires les plus confondues :", ", ".join(f"{p} {n}" for p, n in pairs.head(3).items()))
cm

- **Le réseau ne reconnaît vraiment que le Baroque** : 0,80 d'accuracy, contre 0,17 pour le Cubism et 0,25 pour l'Impressionism, à peine au-dessus d'une réponse au hasard (0,20).
- **Le Baroque est aussi la réponse par défaut** : 48 des 177 œuvres mal classées sont prises pour du Baroque, et 44 pour de l'Ukiyo_e. Deux des trois erreurs les plus fréquentes vont vers le Baroque : Cubism, 18 fois, et Impressionism, 16 fois.
- **La paire la plus confondue est Pop_Art et Ukiyo_e** : 28 erreurs dans les deux sens, dont 17 œuvres de Pop_Art prises pour de l'Ukiyo_e. C'était déjà la principale confusion du réseau de référence de `02_baseline`.

### 4.2 Exemples d'œuvres mal classées

Six œuvres tirées au hasard pour chacune des trois erreurs les plus fréquentes. Les images n'apparaissent que dans les sorties du notebook, qui ne sont pas versionnées.

In [ ]:
N_SHOW = 6
rng = np.random.default_rng(SEED)
top_errors = errors.nlargest(3)
fig, axes = plt.subplots(len(top_errors), N_SHOW, figsize=(1.6 * N_SHOW + 1.4, 1.7 * len(top_errors)))
for row, ((true_style, pred_style), n) in zip(axes, top_errors.items()):
    wrong = np.flatnonzero((val_meta["label"] == true_style) & (val_meta["pred"] == pred_style))
    for ax, i in zip(row, rng.choice(wrong, size=N_SHOW, replace=False)):
        ax.imshow(X_all[splits["val"][i]].permute(1, 2, 0))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    row[0].set_ylabel(f"{true_style}\npris pour\n{pred_style}\n({n} œuvres)", rotation=0, ha="right", va="center")
fig.suptitle("Œuvres de validation mal classées, pour les trois erreurs les plus fréquentes",
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

- **Cubism pris pour Baroque** : des natures mortes, la plupart sur fond sombre et dans des tons bruns.
- **Pop_Art pris pour Ukiyo_e** : surtout des fonds clairs ou blancs, comme le papier des estampes japonaises.
- **Impressionism pris pour Baroque** : plusieurs tableaux sombres ou aux tons terreux, dont un portrait sur fond noir.

Dans les trois cas, l'erreur semble suivre la luminosité et les couleurs d'ensemble plutôt que la touche ou la composition. La section 4.5 le vérifie.

### 4.3 Artistes nommés et artistes inconnus

Le découpage garde dans un seul split les œuvres d'un même artiste nommé, mais chaque œuvre d'artiste inconnu forme son propre groupe : deux œuvres d'un même peintre non identifié peuvent se retrouver l'une en train, l'autre en validation. Si c'était fréquent, les œuvres d'artiste inconnu seraient mieux reconnues que celles d'artiste nommé.

In [ ]:
val_meta["artiste"] = np.where(val_meta["artist"] == UNKNOWN_ARTIST, "inconnu", "nommé")
for kind, group in val_meta.groupby("artiste")["correct"]:
    print(f"Artiste {kind} : {group.sum()} œuvres bien classées sur {len(group)}, accuracy {group.mean():.3f}")
val_meta.groupby(["label", "artiste"])["correct"].agg(["size", "mean"]).round(3)

- **La comparaison globale ne peut pas trancher** : la validation ne compte que 16 œuvres d'artiste nommé, dont 15 cubistes, et aucune en Impressionism, Pop_Art ou Ukiyo_e. L'écart d'accuracy (0,423 pour les artistes inconnus, 0,188 pour les artistes nommés) mélange donc l'effet de l'artiste et celui du style : le Cubism est le style le moins bien reconnu.
- **À style égal, pas d'avantage pour les artistes inconnus** : dans le Cubism, 0,178 sur 45 œuvres d'artiste inconnu, contre 0,133 sur 15 œuvres d'artiste nommé, sur des effectifs trop petits pour conclure. Rien n'indique que des œuvres d'un même peintre inconnu, réparties entre train et validation, facilitent la tâche du réseau.

### 4.4 Genres dominants

Le tableau des genres de `01_donnees` (section 3) montre des styles où un genre pèse lourd : les natures mortes pour le Cubism, les scènes de genre pour l'Ukiyo_e. Accuracy par style et par genre, pour les groupes d'au moins 10 œuvres en validation :

In [ ]:
by_genre = val_meta.groupby(["label", "genre"])["correct"].agg(["size", "mean"]).query("size >= 10")
still_life_pred = val_meta.query("label == 'Cubism' and genre == 'still_life'")["pred"].value_counts()
still_life_train = meta.iloc[splits["train"]].query("genre == 'still_life'")["label"].value_counts()
print("Natures mortes cubistes de la validation, par style prédit :", still_life_pred.to_dict())
print("Natures mortes du train, par style réel :", still_life_train.to_dict())
by_genre.round(3)

- **Le genre est rarement renseigné** : en validation, seuls deux groupes ont un genre connu et au moins 10 œuvres, les natures mortes cubistes et les scènes de genre de l'Ukiyo_e.
- **Les natures mortes cubistes sont presque toujours mal classées** : 0,115 d'accuracy sur 26 œuvres, et 12 sont prises pour du Baroque. Le sujet seul n'explique pas l'erreur, puisque le train compte plus de natures mortes cubistes (57) que baroques (25).
- **Les scènes de genre de l'Ukiyo_e sont bien reconnues** : 0,846 sur 13 œuvres, contre 0,404 pour les autres œuvres d'Ukiyo_e.

### 4.5 Luminosité et couleurs d'ensemble

Hypothèse : sur des pixels aplatis, le réseau s'appuie surtout sur la luminosité et les couleurs d'ensemble de l'image, pas sur sa composition. Deux mesures :

- la luminance moyenne des œuvres, de 0 (noir) à 1 (blanc), par style réel dans le train et par style prédit en validation ;
- une régression logistique qui ne voit que six nombres par image, la moyenne et l'écart-type de chaque canal, avec les réglages par défaut de scikit-learn.

In [ ]:
pixels = X_all.float() / 255
luminance = (0.299 * pixels[:, 0] + 0.587 * pixels[:, 1] + 0.114 * pixels[:, 2]).mean(dim=(1, 2)).numpy()
colour_stats = torch.cat([pixels.mean(dim=(2, 3)), pixels.std(dim=(2, 3))], dim=1).numpy()
del pixels

val_meta["luminance"] = luminance[splits["val"]]
train_meta = meta.iloc[splits["train"]].assign(luminance=luminance[splits["train"]])
by_luminance = pd.DataFrame({
    "train, style réel": train_meta.groupby("label")["luminance"].mean(),
    "validation, style prédit": val_meta.groupby("pred")["luminance"].mean(),
    "erreurs de validation, style prédit": val_meta[~val_meta["correct"]].groupby("pred")["luminance"].mean(),
})

colour_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
colour_model.fit(colour_stats[splits["train"]], y_all[splits["train"]].numpy())
colour_pred = colour_model.predict(colour_stats[splits["val"]])
print(f"Régression logistique sur 6 statistiques de couleur : {(colour_pred == y_val).mean():.3f} en validation, "
      f"contre {(y_pred == y_val).mean():.3f} pour le réseau dense sur {N_IN} pixels")
print(f"Même style prédit que le réseau dense pour {(colour_pred == y_pred).mean():.0%} des œuvres")
cubism = val_meta[val_meta["label"] == "Cubism"]
taken_for_baroque = cubism["pred"] == "Baroque"
print(f"Œuvres cubistes de validation : luminance {cubism.loc[taken_for_baroque, 'luminance'].mean():.3f} "
      f"pour les {taken_for_baroque.sum()} prises pour du Baroque, {cubism.loc[~taken_for_baroque, 'luminance'].mean():.3f} "
      f"pour les autres")
by_luminance.round(3)

- **Le réseau range les images sombres dans le Baroque et les images claires dans l'Ukiyo_e.** Le Baroque est le style le plus sombre du train (luminance 0,416). Les œuvres que le réseau prédit Baroque sont plus sombres encore (0,359), y compris celles qu'il classe à tort (0,386). À l'autre bout, les œuvres prises à tort pour de l'Ukiyo_e ont une luminance de 0,620, au-dessus de celle de l'Ukiyo_e du train (0,555).
- **C'est le cas des natures mortes cubistes** : les 18 œuvres cubistes prises pour du Baroque ont une luminance de 0,384, contre 0,518 pour les autres œuvres cubistes.
- **Six statistiques de couleur font presque aussi bien que les 49 152 pixels** : 0,393 en validation pour la régression logistique, contre 0,410 pour le réseau dense. Les deux modèles ne prédisent pourtant le même style que pour 42 % des œuvres : le réseau capte autre chose que ces six nombres, sans que cela l'aide à reconnaître le style.

### 4.6 Bilan

- **Un seul style bien reconnu, le Baroque** (0,80 d'accuracy), qui sert aussi de réponse par défaut pour les images sombres : 48 des 177 erreurs.
- **Erreur la plus fréquente : Cubism pris pour Baroque** (18 œuvres), surtout des natures mortes sombres. **Paire la plus confondue : Pop_Art et Ukiyo_e** (28 erreurs dans les deux sens).
- **Les artistes inconnus n'expliquent pas les erreurs** : à style égal, dans le Cubism, leurs œuvres ne sont pas mieux reconnues. Ailleurs, la comparaison se confond avec le style, faute d'artistes nommés en validation.
- **Le réseau dense lit surtout la luminosité et les couleurs d'ensemble** : une régression logistique sur six statistiques de couleur atteint presque son score. C'est la limite à présenter en soutenance, et l'argument pour le jalon `m3` : un ResNet pré-entraîné détecte des motifs locaux (touches, contours, aplats), où qu'ils soient dans l'image.